In [1]:
# Author: Till Zemann
# License: MIT License
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch import optim
from tqdm import tqdm

import gymnasium as gym
import pickle as pkl
print(torch.__version__)
from net.a2c import A2C
from net.dale_util import RL_Hidden_Buffer
from rl.rl_util import tag_lunar_lander_phases_batch, tag_lunar_lander_phases_batch_2
import cv2

/cephfs/wangjiyi/miniconda3/envs/mixer/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.12.0


In [2]:
import os
import pickle as pkl
import numpy as np
import torch
import gymnasium as gym
from rl.buffer import ReplayBuffer
def to_one_hot(action, num_actions):
    one_hot = np.zeros(num_actions, dtype=np.float32)
    one_hot[action] = 1.0
    return one_hot
def generate_merged_replay_buffer_from_all_seeds(
    configuration,
    actor_lr,
    critic_lr,
    simple_type,
    ei_type,
    n_seed=1,
    n_episodes=20,
    max_buffer_size=int(1e6),
    hidden_size=64,
    env_name="LunarLander-v3",
    device="cpu",
    save_buffer=True,
):
    """
    Roll out all trained seeds and merge all transitions into one ReplayBuffer.

    Returns
    -------
    replay_buffer : ReplayBuffer
        One merged replay buffer containing data from all seeds.
    reward_ar : np.ndarray
        Shape (n_seed, n_episodes), episode returns.
    """

    info_path = (
        f'./rl/result/lunarlander/simple_{simple_type}_ei_{ei_type}/'
        f'critic_{critic_lr}_actor_{actor_lr}/{configuration}.pkl'
    )

    with open(info_path, 'rb') as f:
        info = pkl.load(f)

    net_state_dicts = info['net_state_dict']

    env = gym.make(env_name, max_episode_steps=500)

    state_dim = env.observation_space.shape[0]

    action_shape = env.action_space.n   # e.g., 4
    action_dim = action_shape           # instead of 1
    print('state_dim:', state_dim, 'action_dim: ', action_dim)

    replay_buffer = ReplayBuffer(
        state_dim=state_dim,
        action_dim=action_dim,
        max_size=max_buffer_size,
        device=device
    )

    reward_ar = np.zeros((n_seed, n_episodes), dtype=np.float32)

    for seed in range(n_seed):
        print(f"\n===== Collecting data from seed {seed} =====")

        torch.manual_seed(seed)
        np.random.seed(seed)

        agent = A2C(
            state_dim,
            action_shape,
            device,
            critic_lr,
            actor_lr,
            1,  # n_envs
            hidden_size,
            configuration,
            simple_type=simple_type,
            ei_type=ei_type
        )
        agent.load_state_dict(net_state_dicts[seed])
        agent.to(device)
        agent.eval()

        for episode in range(n_episodes):
            state, info_env = env.reset(seed=seed * 1000 + episode)
            done = False
            ep_reward = 0.0

            hidden = (
                torch.zeros((1, hidden_size), device=device),
                torch.zeros((1, hidden_size), device=device)
            )

            while not done:
                with torch.no_grad():
                    action, _, _, _, new_hidden = agent.select_action(state[None, :], hidden)

                env_action = int(action.item())
                next_state, reward, terminated, truncated, info_step = env.step(env_action)
                done = terminated or truncated

                # single-task placeholder
                task = np.array([0.0], dtype=np.float32)
                next_task = np.array([0.0], dtype=np.float32)

                replay_buffer.add(
                    state=np.asarray(state, dtype=np.float32),
                    action=to_one_hot(env_action, action_shape),
                    next_state=np.asarray(next_state, dtype=np.float32),
                    reward=np.asarray([reward], dtype=np.float32),
                    done=np.asarray([float(done)], dtype=np.float32),
                    task=task,
                    next_task=next_task,
                )

                ep_reward += reward
                state = next_state

                h_critic, h_actor = new_hidden
                if done:
                    h_critic = torch.zeros_like(h_critic)
                    h_actor = torch.zeros_like(h_actor)
                hidden = (h_critic, h_actor)

            reward_ar[seed, episode] = ep_reward
            print(
                f"Seed {seed}, episode {episode}, reward: {ep_reward:.2f}, "
                f"merged buffer size: {replay_buffer.size}"
            )

    if save_buffer:
        merged_buffer_path = info_path.replace('.pkl', '_merged_replay_buffer.pkl')
        os.makedirs(os.path.dirname(merged_buffer_path), exist_ok=True)
        with open(merged_buffer_path, 'wb') as f:
            pkl.dump(replay_buffer.state_dict(), f)
        print(f"\nSaved merged replay buffer to: {merged_buffer_path}, size: ", replay_buffer.size)

    env.close()
    return replay_buffer, reward_ar

ModuleNotFoundError: No module named 'rl.buffer'

In [4]:
actor_lrs = [0.003]
critic_lrs = [0.005]
configurations = ['eisep_all']

n_episodes = 40
n_seed = 5
simple_type = 'rnn'
ei_type = 'eisep'

all_reward_ars = np.zeros(
    (len(actor_lrs), len(critic_lrs), len(configurations), n_seed, n_episodes),
    dtype=np.float32
)

all_merged_buffers = {}

for a, actor_lr in enumerate(actor_lrs):
    for c, critic_lr in enumerate(critic_lrs):
        for i, configuration in enumerate(configurations):
            print(f"\nconfiguration={configuration}, actor_lr={actor_lr}, critic_lr={critic_lr}")

            merged_buffer, reward_ar = generate_merged_replay_buffer_from_all_seeds(
                configuration=configuration,
                actor_lr=actor_lr,
                critic_lr=critic_lr,
                simple_type=simple_type,
                ei_type=ei_type,
                n_seed=n_seed,
                n_episodes=n_episodes,
                max_buffer_size=200000,
                hidden_size=64,
                env_name="LunarLander-v3",
                device="cpu",
                save_buffer=True,
            )

            all_reward_ars[a, c, i] = reward_ar
            all_merged_buffers[(configuration, actor_lr, critic_lr)] = merged_buffer


configuration=eisep_all, actor_lr=0.003, critic_lr=0.005
state_dim: 8 action_dim:  4

===== Collecting data from seed 0 =====
Using <class 'net.rnncell_variant.EIRNNCell'> for actor and critic
Seed 0, episode 0, reward: 258.72, merged buffer size: 336
Seed 0, episode 1, reward: 242.28, merged buffer size: 705
Seed 0, episode 2, reward: 223.09, merged buffer size: 1100
Seed 0, episode 3, reward: 215.78, merged buffer size: 1488
Seed 0, episode 4, reward: 228.39, merged buffer size: 1926
Seed 0, episode 5, reward: 244.30, merged buffer size: 2336
Seed 0, episode 6, reward: 285.27, merged buffer size: 2723
Seed 0, episode 7, reward: 242.85, merged buffer size: 3100


KeyboardInterrupt: 

In [ ]:

import os
import pickle as pkl
import numpy as np
import torch


def normalize_and_split_replay_buffer(
    replay_buffer=None,
    replay_buffer_path=None,
    state_dim=None,
    action_dim=None,
    scale_factor=0.2,
    train_ratio=0.8,
    shuffle=True,
    seed=0,
    zero_reward_done=False,
    save_dir=None,
    save_as_torch=True,
    prefix="replay_buffer_scaled5",
):
    """
    Normalize a replay buffer by dividing state/action/next_state by scale_factor,
    then split into train/test with ratio train_ratio:(1-train_ratio).

    Parameters
    ----------
    replay_buffer : ReplayBuffer or None
        In-memory ReplayBuffer object.
    replay_buffer_path : str or None
        Path to saved replay buffer state_dict (.pkl or .pth).
    state_dim : int or None
        Required only if loading from state_dict and reconstructing ReplayBuffer is needed.
        Not necessary for this function if only using raw arrays from state_dict.
    action_dim : int or None
        Same note as state_dim.
    scale_factor : float
        Divide state/action/next_state by this value.
    train_ratio : float
        Fraction used for training set.
    shuffle : bool
        Whether to shuffle before split.
    seed : int
        Random seed for reproducibility.
    zero_reward_done : bool
        If True, set reward and done to zero in the split buffers.
        If False, preserve original reward and done.
    save_dir : str or None
        If provided, save train/test buffers to this directory.
    save_as_torch : bool
        If True, save with torch.save(... .pth).
        If False, save with pickle(... .pkl).
    prefix : str
        Prefix of saved filenames.

    Returns
    -------
    train_buffer_state_dict : dict
    test_buffer_state_dict : dict
    """

    # ------------------------------------------------------------
    # Load replay buffer data
    # ------------------------------------------------------------
    if replay_buffer is not None:
        original_state_dict = replay_buffer.state_dict()
    elif replay_buffer_path is not None:
        if replay_buffer_path.endswith(".pth"):
            original_state_dict = torch.load(replay_buffer_path, map_location="cpu")
        else:
            with open(replay_buffer_path, "rb") as f:
                original_state_dict = pkl.load(f)
    else:
        raise ValueError("Please provide either replay_buffer or replay_buffer_path.")

    # Convert tensors to numpy if needed
    def to_numpy(x):
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
        return np.asarray(x)

    original_state = to_numpy(original_state_dict["state"])
    original_action = to_numpy(original_state_dict["action"])
    original_next_state = to_numpy(original_state_dict["next_state"])
    original_reward = to_numpy(original_state_dict["reward"])
    original_done = to_numpy(original_state_dict["done"])
    original_task = to_numpy(original_state_dict["task"])
    original_next_task = to_numpy(original_state_dict["next_task"])

    length = int(original_state_dict["size"])
    print('Buffer size:', length)

    # Safety: truncate to actual size
    original_state = original_state[:length]
    original_action = original_action[:length]
    original_next_state = original_next_state[:length]
    original_reward = original_reward[:length]
    original_done = original_done[:length]
    original_task = original_task[:length]
    original_next_task = original_next_task[:length]

    # ------------------------------------------------------------
    # Normalize by scaling only
    # ------------------------------------------------------------
    normalized_state = original_state / scale_factor
    normalized_action = original_action
    normalized_next_state = original_next_state / scale_factor

    # ------------------------------------------------------------
    # Train/test split
    # ------------------------------------------------------------
    indices = np.arange(length)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    split = int(length * train_ratio)
    train_idx = indices[:split]
    test_idx = indices[split:]

    # ------------------------------------------------------------
    # Build train set
    # ------------------------------------------------------------
    train_state = torch.FloatTensor(normalized_state[train_idx])
    train_action = torch.FloatTensor(normalized_action[train_idx])
    train_next_state = torch.FloatTensor(normalized_next_state[train_idx])
    train_task = torch.FloatTensor(original_task[train_idx])
    train_next_task = torch.FloatTensor(original_next_task[train_idx])

    if zero_reward_done:
        train_reward = torch.zeros((len(train_idx), 1), dtype=torch.float32)
        train_done = torch.zeros((len(train_idx), 1), dtype=torch.float32)
    else:
        train_reward = torch.FloatTensor(original_reward[train_idx])
        train_done = torch.FloatTensor(original_done[train_idx])

    train_buffer_state_dict = {
        "state": train_state,
        "action": train_action,
        "next_state": train_next_state,
        "reward": train_reward,
        "done": train_done,
        "task": train_task,
        "next_task": train_next_task,
        "size": len(train_idx),
        "ptr": 0,
    }

    # ------------------------------------------------------------
    # Build test set
    # ------------------------------------------------------------
    test_state = torch.FloatTensor(normalized_state[test_idx])
    test_action = torch.FloatTensor(normalized_action[test_idx])
    test_next_state = torch.FloatTensor(normalized_next_state[test_idx])
    test_task = torch.FloatTensor(original_task[test_idx])
    test_next_task = torch.FloatTensor(original_next_task[test_idx])

    if zero_reward_done:
        test_reward = torch.zeros((len(test_idx), 1), dtype=torch.float32)
        test_done = torch.zeros((len(test_idx), 1), dtype=torch.float32)
    else:
        test_reward = torch.FloatTensor(original_reward[test_idx])
        test_done = torch.FloatTensor(original_done[test_idx])

    test_buffer_state_dict = {
        "state": test_state,
        "action": test_action,
        "next_state": test_next_state,
        "reward": test_reward,
        "done": test_done,
        "task": test_task,
        "next_task": test_next_task,
        "size": len(test_idx),
        "ptr": 0,
    }

    # ------------------------------------------------------------
    # Save if requested
    # ------------------------------------------------------------
    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)

        if save_as_torch:
            train_path = os.path.join(save_dir, f"{prefix}_train.pth")
            test_path = os.path.join(save_dir, f"{prefix}_test.pth")
            torch.save(train_buffer_state_dict, train_path)
            torch.save(test_buffer_state_dict, test_path)
        else:
            train_path = os.path.join(save_dir, f"{prefix}_train.pkl")
            test_path = os.path.join(save_dir, f"{prefix}_test.pkl")
            with open(train_path, "wb") as f:
                pkl.dump(train_buffer_state_dict, f)
            with open(test_path, "wb") as f:
                pkl.dump(test_buffer_state_dict, f)

        print(f"Saved train buffer to: {train_path}")
        print(f"Saved test buffer to: {test_path}")

    print("Original length:", length)
    print("Train size:", len(train_idx))
    print("Test size:", len(test_idx))
    print("State shape:", train_state.shape, test_state.shape)
    print("Action shape:", train_action.shape, test_action.shape)

    return train_buffer_state_dict, test_buffer_state_dict

merged_buffer = all_merged_buffers[('eisep_all', 0.003, 0.005)]
print(merged_buffer.state.mean(), merged_buffer.state.std())
train_buffer_state_dict, test_buffer_state_dict = normalize_and_split_replay_buffer(
    replay_buffer=merged_buffer,
    scale_factor=0.4,
    train_ratio=0.8,
    shuffle=True,
    seed=0,
    zero_reward_done=False,   # set True if you want to mimic your reference exactly
    save_dir="./rl/result/lunarlander/normalized_buffers_lunarlander",
    save_as_torch=True,
    prefix="eisep_all_critic0.005_actor0.001_scaled0.2"
)
print(train_buffer_state_dict['state'].mean(), train_buffer_state_dict['state'].std())

NameError: name 'all_merged_buffers' is not defined

In [18]:
import os
# Headless rendering: prevents pygame segfault when no display is available
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')

class ClipReplayBuffer:
    """
    Replay buffer that stores fixed-length clips with frames.
    Shape: (n_clips, clip_len, feature_dim)
    """
    def __init__(self, n_clips, clip_len, state_dim, action_dim, frame_shape):
        self.n_clips = n_clips
        self.clip_len = clip_len
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.frame_shape = frame_shape
        
        # Initialize storage
        self.state = np.zeros((n_clips, clip_len, state_dim), dtype=np.float32)
        self.action = np.zeros((n_clips, clip_len, action_dim), dtype=np.float32)
        self.reward = np.zeros((n_clips, clip_len, 1), dtype=np.float32)
        self.done = np.zeros((n_clips, clip_len, 1), dtype=np.float32)
        self.next_state = np.zeros((n_clips, clip_len, state_dim), dtype=np.float32)
        self.frame = np.zeros((n_clips, clip_len, *frame_shape), dtype=np.uint8)
        
        self.ptr = 0  # Current clip index
        self.size = 0  # Number of clips filled
        
    def add_clip(self, states, actions, rewards, dones, next_states, frames):
        """Add a complete clip to the buffer."""
        assert states.shape[0] == self.clip_len
        assert next_states.shape[0] == self.clip_len
        assert actions.shape[0] == self.clip_len
        assert rewards.shape[0] == self.clip_len
        assert dones.shape[0] == self.clip_len
        assert frames.shape[0] == self.clip_len
        
        self.state[self.ptr] = states
        self.next_state[self.ptr] = next_states
        self.action[self.ptr] = actions
        self.reward[self.ptr] = rewards
        self.done[self.ptr] = dones
        self.frame[self.ptr] = frames
        
        self.ptr = (self.ptr + 1) % self.n_clips
        self.size = min(self.size + 1, self.n_clips)
        
    def save(self, path):
        np.savez_compressed(
            path,
            state=self.state[:self.size],
            next_state=self.next_state[:self.size],
            action=self.action[:self.size],
            reward=self.reward[:self.size],
            done=self.done[:self.size],
            frame=self.frame[:self.size],
            size=self.size
        )
        print(f"Saved {self.size} clips to {path}")
        
    def load(self, path):
        data = np.load(path)
        self.state[:data['size']] = data['state']
        self.next_state[:data['size']] = data['next_state']
        self.action[:data['size']] = data['action']
        self.reward[:data['size']] = data['reward']
        self.done[:data['size']] = data['done']
        self.frame[:data['size']] = data['frame']
        self.size = int(data['size'])
        self.ptr = self.size % self.n_clips
        print(f"Loaded {self.size} clips from {path}")


def generate_clip_replay_buffer(
    configuration,
    actor_lr,
    critic_lr,
    simple_type,
    ei_type,
    n_clips=200,
    clip_len=100,
    n_seed=1,
    hidden_size=64,
    env_name="LunarLander-v3",
    device="cpu",
    frame_scale=4,
    save_buffer=True,
):
    """
    Generate a replay buffer with fixed-length clips including frames.
    
    Returns
    -------
    clip_buffer : ClipReplayBuffer
        Buffer with shape (n_clips, clip_len, feature_dim)
    """
    info_path = (
        f'./rl/result/lunarlander/simple_{simple_type}_ei_{ei_type}/'
        f'critic_{critic_lr}_actor_{actor_lr}/{configuration}.pkl'
    )
    
    with open(info_path, 'rb') as f:
        info = pkl.load(f)
    
    net_state_dicts = info['net_state_dict']
    
    env = gym.make(env_name, max_episode_steps=500, render_mode='rgb_array')
    
    state_dim = env.observation_space.shape[0]
    action_shape = env.action_space.n
    
    # Get frame shape
    state, _ = env.reset()
    frame = env.render()
    H_new = frame.shape[0] // frame_scale
    W_new = frame.shape[1] // frame_scale
    frame_shape = (H_new, W_new, frame.shape[2])
    
    print(f'state_dim: {state_dim}, action_dim: {action_shape}, frame_shape: {frame_shape}')
    
    clip_buffer = ClipReplayBuffer(
        n_clips=n_clips,
        clip_len=clip_len,
        state_dim=state_dim,
        action_dim=action_shape,
        frame_shape=frame_shape
    )
    
    clips_collected = 0


    n_episode_per_seed = n_clips // n_seed + 1
    for seed in range(n_seed):
        if clips_collected >= n_clips:
            break
            
        print(f"\n===== Collecting clips from seed {seed} =====")
        
        torch.manual_seed(seed)
        np.random.seed(seed)
        
        agent = A2C(
            state_dim,
            action_shape,
            device,
            critic_lr,
            actor_lr,
            1,
            hidden_size,
            configuration,
            simple_type=simple_type,
            ei_type=ei_type
        )
        agent.load_state_dict(net_state_dicts[seed])
        agent.to(device)
        agent.eval()
        

        for episode in range(n_episode_per_seed):
            state, _ = env.reset(seed=seed * 100 + episode)
            done = False
            
            hidden = (
                torch.zeros((1, hidden_size), device=device),
                torch.zeros((1, hidden_size), device=device)
            )
            
            # Temporary storage for current episode
            ep_states = []
            ep_actions = []
            ep_rewards = []
            ep_dones = []
            ep_frames = []
            ep_next_states = []

            while not done:
                # Render and resize frame
                frame_raw = env.render()
                frame_resized = cv2.resize(frame_raw, (W_new, H_new), interpolation=cv2.INTER_AREA)
                
                with torch.no_grad():
                    action, _, _, _, new_hidden = agent.select_action(state[None, :], hidden)
                
                env_action = int(action.item())
                next_state, reward, terminated, truncated, _ = env.step(env_action)
                done = terminated or truncated
                
                # Store transition
                ep_states.append(state)
                ep_actions.append(np.eye(action_shape)[env_action])
                ep_rewards.append([reward])
                ep_dones.append([float(done)])
                ep_next_states.append(next_state)
                ep_frames.append(frame_resized)
                
                state = next_state
                
                h_critic, h_actor = new_hidden
                if done:
                    h_critic = torch.zeros_like(h_critic)
                    h_actor = torch.zeros_like(h_actor)
                hidden = (h_critic, h_actor)
            
            # Convert episode to clips
            ep_len = len(ep_states)
            
            if ep_len >= clip_len:
                start_idx = np.random.randint(ep_len-clip_len)
                end_idx = start_idx + clip_len
                
                    
                clip_states = np.array(ep_states[start_idx:end_idx], dtype=np.float32)
                clip_actions = np.array(ep_actions[start_idx:end_idx], dtype=np.float32)
                clip_rewards = np.array(ep_rewards[start_idx:end_idx], dtype=np.float32)
                clip_dones = np.array(ep_dones[start_idx:end_idx], dtype=np.float32)
                clip_next_states = np.array(ep_next_states[start_idx:end_idx], dtype=np.float32)
                clip_frames = np.array(ep_frames[start_idx:end_idx], dtype=np.uint8)
                    
                clip_buffer.add_clip(clip_states, clip_actions, clip_rewards, clip_dones, clip_next_states, clip_frames)
                clips_collected += 1
            
            print(f"Seed {seed}, episode {episode}, ep_len: {ep_len}, clips: {clips_collected}/{n_clips}")
    
    if save_buffer:
        buffer_path = info_path.replace('.pkl', f'_clip_buffer_n{n_clips}_len{clip_len}.npz')
        os.makedirs(os.path.dirname(buffer_path), exist_ok=True)
        clip_buffer.save(buffer_path)
    
    env.close()
    return clip_buffer

In [19]:
# Test clip buffer generation
configuration = 'eisep_all'
actor_lr = 0.003
critic_lr = 0.005
simple_type = 'rnn'
ei_type = 'eisep'

clip_buffer = generate_clip_replay_buffer(
    configuration=configuration,
    actor_lr=actor_lr,
    critic_lr=critic_lr,
    simple_type=simple_type,
    ei_type=ei_type,
    n_clips=50,
    clip_len=150,
    n_seed=5,
    hidden_size=64,
    env_name="LunarLander-v3",
    device="cpu",
    frame_scale=4,
    save_buffer=True,
)

print(f"\nBuffer shapes:")
print(f"  state: {clip_buffer.state.shape}")
print(f"  action: {clip_buffer.action.shape}")
print(f"  reward: {clip_buffer.reward.shape}")
print(f"  done: {clip_buffer.done.shape}")
print(f"  frame: {clip_buffer.frame.shape}")
print(f"  size: {clip_buffer.size}")

state_dim: 8, action_dim: 4, frame_shape: (100, 150, 3)

===== Collecting clips from seed 0 =====
Using <class 'net.rnncell_variant.EIRNNCell'> for actor and critic
Seed 0, episode 0, ep_len: 336, clips: 1/50
Seed 0, episode 1, ep_len: 369, clips: 2/50
Seed 0, episode 2, ep_len: 395, clips: 3/50
Seed 0, episode 3, ep_len: 388, clips: 4/50
Seed 0, episode 4, ep_len: 438, clips: 5/50
Seed 0, episode 5, ep_len: 410, clips: 6/50
Seed 0, episode 6, ep_len: 387, clips: 7/50
Seed 0, episode 7, ep_len: 377, clips: 8/50
Seed 0, episode 8, ep_len: 331, clips: 9/50
Seed 0, episode 9, ep_len: 395, clips: 10/50
Seed 0, episode 10, ep_len: 392, clips: 11/50

===== Collecting clips from seed 1 =====
Using <class 'net.rnncell_variant.EIRNNCell'> for actor and critic
Seed 1, episode 0, ep_len: 500, clips: 12/50
Seed 1, episode 1, ep_len: 259, clips: 13/50
Seed 1, episode 2, ep_len: 500, clips: 14/50
Seed 1, episode 3, ep_len: 290, clips: 15/50
Seed 1, episode 4, ep_len: 350, clips: 16/50
Seed 1, episod